# Sample 100k Active Companies by Account Category

This notebook reads the full company CSV, filters to active companies, excludes weak/low-frequency account categories, then randomly samples 100,000 companies proportionally by `Accounts_AccountCategory`.

The output keeps all original CSV columns. Only company rows are filtered/sampled.

## 1. Configuration

In [4]:
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

# Change this if your full CSV is stored elsewhere.
INPUT_CSV = Path(r"E:\\000硕士毕设\\公司选取\\UKcompanies_8_sectors_cleaned.csv")
OUTPUT_DIR = Path(r"E:\\000硕士毕设\\公司选取")
OUTPUT_CSV = OUTPUT_DIR / "UKcompanies_active_account_category_sample_100k.csv"
OUTPUT_DISTRIBUTION_CSV = OUTPUT_DIR / "UKcompanies_active_account_category_sample_100k_distribution.csv"

COMPANY_STATUS_COL = "CompanyStatus"
ACCOUNT_CATEGORY_COL = "Accounts_AccountCategory"

ACTIVE_STATUS_VALUES = {"ACTIVE"}
EXCLUDE_ACCOUNT_CATEGORIES = {
    "DORMANT",
    "NO ACCOUNTS FILED",
    "AUDITED ABRIDGED",
    "TOTAL EXEMPTION SMALL",
    "FILING EXEMPTION SUBSIDIARY",
    "ACCOUNTS TYPE NOT AVAILABLE",
    "PARTIAL EXEMPTION",
}

TARGET_SAMPLE_SIZE = 100_000
RANDOM_SEED = 20260707
CHUNKSIZE = 200_000
CSV_ENCODING_CANDIDATES = ["utf-8-sig", "utf-8", "gb18030"]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input CSV:", INPUT_CSV)
print("Output sample CSV:", OUTPUT_CSV)
print("Output distribution CSV:", OUTPUT_DISTRIBUTION_CSV)
print("Target sample size:", TARGET_SAMPLE_SIZE)
print("Active status values:", ACTIVE_STATUS_VALUES)
print("Excluded account categories:", EXCLUDE_ACCOUNT_CATEGORIES)

Input CSV: E:\000硕士毕设\公司选取\UKcompanies_8_sectors_cleaned.csv
Output sample CSV: E:\000硕士毕设\公司选取\UKcompanies_active_account_category_sample_100k.csv
Output distribution CSV: E:\000硕士毕设\公司选取\UKcompanies_active_account_category_sample_100k_distribution.csv
Target sample size: 100000
Active status values: {'ACTIVE'}
Excluded account categories: {'NO ACCOUNTS FILED', 'AUDITED ABRIDGED', 'ACCOUNTS TYPE NOT AVAILABLE', 'TOTAL EXEMPTION SMALL', 'PARTIAL EXEMPTION', 'DORMANT', 'FILING EXEMPTION SUBSIDIARY'}


## 2. Helper Functions

In [5]:
def find_working_encoding(path, encodings):
    last_error = None
    for enc in encodings:
        try:
            pd.read_csv(path, nrows=5, encoding=enc)
            return enc
        except Exception as exc:
            last_error = exc
    raise last_error


def clean_upper(s):
    return s.astype("string").str.strip().str.upper()


def largest_remainder_counts(counts, target_total):
    total = sum(counts.values())
    if total <= 0:
        raise ValueError("No eligible rows available for sampling.")
    raw = {k: v / total * target_total for k, v in counts.items()}
    base = {k: int(np.floor(x)) for k, x in raw.items()}
    remainder = target_total - sum(base.values())
    order = sorted(raw, key=lambda k: raw[k] - base[k], reverse=True)
    for k in order[:remainder]:
        base[k] += 1
    return base


def filter_eligible(chunk):
    status = clean_upper(chunk[COMPANY_STATUS_COL])
    category = clean_upper(chunk[ACCOUNT_CATEGORY_COL])
    mask = status.isin(ACTIVE_STATUS_VALUES) & ~category.isin(EXCLUDE_ACCOUNT_CATEGORIES)
    return chunk.loc[mask].copy()


def category_key(series):
    return series.astype("string").str.strip().str.upper()

## 3. First Pass: Count Eligible Active Companies by Account Category

In [6]:
if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")

encoding = find_working_encoding(INPUT_CSV, CSV_ENCODING_CANDIDATES)
print("Using encoding:", encoding)

header = pd.read_csv(INPUT_CSV, nrows=0, encoding=encoding)
columns = list(header.columns)
required_cols = {COMPANY_STATUS_COL, ACCOUNT_CATEGORY_COL}
missing_required = required_cols - set(columns)
if missing_required:
    print("Available columns:")
    for c in columns:
        print(" -", c)
    raise KeyError(f"Missing required columns: {missing_required}")

total_rows = 0
active_rows = 0
eligible_rows = 0
excluded_category_rows_after_active = 0
category_counts = Counter()

for i, chunk in enumerate(
    pd.read_csv(
        INPUT_CSV,
        usecols=[COMPANY_STATUS_COL, ACCOUNT_CATEGORY_COL],
        chunksize=CHUNKSIZE,
        encoding=encoding,
        low_memory=False,
    ),
    start=1,
):
    total_rows += len(chunk)
    status = clean_upper(chunk[COMPANY_STATUS_COL])
    category = clean_upper(chunk[ACCOUNT_CATEGORY_COL])
    active_mask = status.isin(ACTIVE_STATUS_VALUES)
    active_rows += int(active_mask.sum())
    excluded_category_rows_after_active += int((active_mask & category.isin(EXCLUDE_ACCOUNT_CATEGORIES)).sum())
    eligible_category = category[active_mask & ~category.isin(EXCLUDE_ACCOUNT_CATEGORIES)]
    eligible_rows += int(eligible_category.shape[0])
    category_counts.update(eligible_category.dropna().tolist())
    print(f"First pass chunk {i:,}; rows scanned: {total_rows:,}")

print("Done first pass.")
print("Total rows:", f"{total_rows:,}")
print("Active rows:", f"{active_rows:,}")
print("Excluded account category rows after Active filter:", f"{excluded_category_rows_after_active:,}")
print("Eligible rows:", f"{eligible_rows:,}")

Using encoding: utf-8-sig
First pass chunk 1; rows scanned: 200,000
First pass chunk 2; rows scanned: 400,000
First pass chunk 3; rows scanned: 600,000
First pass chunk 4; rows scanned: 800,000
First pass chunk 5; rows scanned: 1,000,000
First pass chunk 6; rows scanned: 1,200,000
First pass chunk 7; rows scanned: 1,400,000
First pass chunk 8; rows scanned: 1,600,000
First pass chunk 9; rows scanned: 1,800,000
First pass chunk 10; rows scanned: 2,000,000
First pass chunk 11; rows scanned: 2,200,000
First pass chunk 12; rows scanned: 2,400,000
First pass chunk 13; rows scanned: 2,600,000
First pass chunk 14; rows scanned: 2,800,000
First pass chunk 15; rows scanned: 3,000,000
First pass chunk 16; rows scanned: 3,200,000
First pass chunk 17; rows scanned: 3,400,000
First pass chunk 18; rows scanned: 3,415,689
Done first pass.
Total rows: 3,415,689
Active rows: 3,152,289
Excluded account category rows after Active filter: 1,125,236
Eligible rows: 2,027,053


## 4. Calculate Proportional Category Quotas

In [7]:
if eligible_rows < TARGET_SAMPLE_SIZE:
    raise ValueError(f"Eligible rows ({eligible_rows:,}) are fewer than target sample size ({TARGET_SAMPLE_SIZE:,}).")

quota_by_category = largest_remainder_counts(category_counts, TARGET_SAMPLE_SIZE)

distribution = pd.DataFrame([
    {
        ACCOUNT_CATEGORY_COL: category,
        "eligible_count": count,
        "eligible_percent": count / eligible_rows,
        "sample_quota": quota_by_category[category],
        "sample_percent": quota_by_category[category] / TARGET_SAMPLE_SIZE,
    }
    for category, count in category_counts.most_common()
])

print("Quota sum:", distribution["sample_quota"].sum())
distribution

Quota sum: 100000


,Accounts_AccountCategory,eligible_count,eligible_percent,sample_quota,sample_percent
0,MICRO ENTITY,1079219,0.532408,53241,0.53241
1,TOTAL EXEMPTION FULL,748335,0.369174,36917,0.36917
2,UNAUDITED ABRIDGED,91523,0.045151,4515,0.04515
3,FULL,37182,0.018343,1834,0.01834
4,SMALL,36109,0.017814,1781,0.01781
5,AUDIT EXEMPTION SUBSIDIARY,18274,0.009015,902,0.00902
6,GROUP,12698,0.006264,627,0.00627
7,MEDIUM,3713,0.001832,183,0.00183


## 5. Second Pass: Stratified Random Sampling

This pass keeps all original columns. It assigns each eligible row a random priority and keeps the lowest random priorities within each account category quota. This is equivalent to random sampling without loading the full dataset into memory.

In [8]:
rng = np.random.default_rng(RANDOM_SEED)
selected_by_category = {}
rows_seen_eligible_second_pass = 0

for i, chunk in enumerate(
    pd.read_csv(
        INPUT_CSV,
        chunksize=CHUNKSIZE,
        encoding=encoding,
        low_memory=False,
    ),
    start=1,
):
    eligible = filter_eligible(chunk)
    if eligible.empty:
        print(f"Second pass chunk {i:,}; no eligible rows")
        continue

    rows_seen_eligible_second_pass += len(eligible)
    eligible["__account_category_key"] = category_key(eligible[ACCOUNT_CATEGORY_COL])
    eligible["__random_priority"] = rng.random(len(eligible))

    for category, part in eligible.groupby("__account_category_key", sort=False):
        quota = quota_by_category.get(category, 0)
        if quota <= 0:
            continue
        if category in selected_by_category:
            combined = pd.concat([selected_by_category[category], part], ignore_index=True)
        else:
            combined = part
        selected_by_category[category] = combined.nsmallest(quota, "__random_priority")

    current_selected = sum(len(df) for df in selected_by_category.values())
    print(f"Second pass chunk {i:,}; eligible rows seen: {rows_seen_eligible_second_pass:,}; selected kept: {current_selected:,}")

sample = pd.concat(selected_by_category.values(), ignore_index=True)
sample = sample.drop(columns=["__account_category_key", "__random_priority"])

# Shuffle final output order while keeping reproducibility.
sample = sample.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print("Final sample rows:", f"{len(sample):,}")
sample.head()

Second pass chunk 1; eligible rows seen: 166,562; selected kept: 100,000
Second pass chunk 2; eligible rows seen: 333,086; selected kept: 100,000
Second pass chunk 3; eligible rows seen: 497,216; selected kept: 100,000
Second pass chunk 4; eligible rows seen: 659,493; selected kept: 100,000
Second pass chunk 5; eligible rows seen: 817,307; selected kept: 100,000
Second pass chunk 6; eligible rows seen: 970,672; selected kept: 100,000
Second pass chunk 7; eligible rows seen: 1,117,289; selected kept: 100,000
Second pass chunk 8; eligible rows seen: 1,268,720; selected kept: 100,000
Second pass chunk 9; eligible rows seen: 1,424,715; selected kept: 100,000
Second pass chunk 10; eligible rows seen: 1,579,650; selected kept: 100,000
Second pass chunk 11; eligible rows seen: 1,727,841; selected kept: 100,000
Second pass chunk 12; eligible rows seen: 1,858,162; selected kept: 100,000
Second pass chunk 13; eligible rows seen: 1,893,995; selected kept: 100,000
Second pass chunk 14; eligible ro

,CompanyNumber,CompanyName,CompanyStatus,CompanyCategory,CountryOfOrigin,RegAddress_Country,RegAddress_PostTown,RegAddress_PostCode,IncorporationDate,primary_sic_code,...,sector_id,Accounts_AccountCategory,Accounts_LastMadeUpDate,Accounts_NextDueDate,Mortgages_NumMortOutstanding,Mortgages_NumMortCharges,has_outstanding_charges,company_age_years,CountryOfOrigin_clean,is_uk_company
0,13209628,TICKETY BOO TEETH LIMITED,Active,Private Limited Company,United Kingdom,UNITED KINGDOM,BASINGSTOKE,RG24 8PE,2021-02-18,86230,...,6.0,UNAUDITED ABRIDGED,2025-06-30,2027-03-31,0,0,False,5.3,united kingdom,True
1,13887674,TECH INFUSION LIMITED,Active,Private Limited Company,United Kingdom,ENGLAND,HALIFAX,HX1 5LT,2022-02-02,62020,...,8.0,TOTAL EXEMPTION FULL,2025-02-28,2026-11-30,0,0,False,4.4,united kingdom,True
2,13408899,THE SCC ACADEMY LIMITED,Active,Private Limited Company,United Kingdom,ENGLAND,WARWICKSHIRE,CV37 6YX,2021-05-19,62090,...,8.0,TOTAL EXEMPTION FULL,2025-04-05,2027-01-05,0,0,False,5.1,united kingdom,True
3,SC374368,GRACEFRUIT LIMITED,Active,Private Limited Company,United Kingdom,NaN,LONGCROFT,FK4 1QL,2010-03-08,46450,...,3.0,TOTAL EXEMPTION FULL,2025-03-31,2026-12-31,0,0,False,16.3,united kingdom,True
4,13675979,LINHAM LIMITED,Active,Private Limited Company,United Kingdom,UNITED KINGDOM,HOLYWELL,CH8 7LH,2021-10-13,47110,...,3.0,TOTAL EXEMPTION FULL,2025-10-31,2027-07-31,1,1,True,4.7,united kingdom,True


## 6. Validate Sample Distribution

In [9]:
sample_distribution = (
    sample[ACCOUNT_CATEGORY_COL]
    .astype("string")
    .str.strip()
    .str.upper()
    .value_counts()
    .rename_axis(ACCOUNT_CATEGORY_COL)
    .reset_index(name="sample_count")
)
sample_distribution["sample_percent_actual"] = sample_distribution["sample_count"] / len(sample)

validation = distribution.merge(sample_distribution, on=ACCOUNT_CATEGORY_COL, how="left")
validation["sample_count"] = validation["sample_count"].fillna(0).astype(int)
validation["sample_percent_actual"] = validation["sample_percent_actual"].fillna(0)
validation["quota_minus_actual"] = validation["sample_quota"] - validation["sample_count"]

print("CompanyStatus values in final sample:")
display(sample[COMPANY_STATUS_COL].value_counts(dropna=False).reset_index(name="count"))

print("Excluded category check in final sample:")
display(sample[sample[ACCOUNT_CATEGORY_COL].astype("string").str.strip().str.upper().isin(EXCLUDE_ACCOUNT_CATEGORIES)][[COMPANY_STATUS_COL, ACCOUNT_CATEGORY_COL]].head())

validation

CompanyStatus values in final sample:


,CompanyStatus,count
0,Active,100000


Excluded category check in final sample:


,CompanyStatus,Accounts_AccountCategory


,Accounts_AccountCategory,eligible_count,eligible_percent,sample_quota,sample_percent,sample_count,sample_percent_actual,quota_minus_actual
0,MICRO ENTITY,1079219,0.532408,53241,0.53241,53241,0.53241,0
1,TOTAL EXEMPTION FULL,748335,0.369174,36917,0.36917,36917,0.36917,0
2,UNAUDITED ABRIDGED,91523,0.045151,4515,0.04515,4515,0.04515,0
3,FULL,37182,0.018343,1834,0.01834,1834,0.01834,0
4,SMALL,36109,0.017814,1781,0.01781,1781,0.01781,0
5,AUDIT EXEMPTION SUBSIDIARY,18274,0.009015,902,0.00902,902,0.00902,0
6,GROUP,12698,0.006264,627,0.00627,627,0.00627,0
7,MEDIUM,3713,0.001832,183,0.00183,183,0.00183,0


## 7. Write Output Files

In [11]:
sample.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
validation.to_csv(OUTPUT_DISTRIBUTION_CSV, index=False, encoding="utf-8-sig")

print("Sample CSV written:", OUTPUT_CSV)
print("Distribution validation CSV written:", OUTPUT_DISTRIBUTION_CSV)
print("Rows written:", f"{len(sample):,}")

Sample CSV written: E:\000硕士毕设\公司选取\UKcompanies_active_account_category_sample_100k.csv
Distribution validation CSV written: E:\000硕士毕设\公司选取\UKcompanies_active_account_category_sample_100k_distribution.csv
Rows written: 100,000
